# 00 Run Everything - Publication Workflow

This is the single notebook entry point. Run this first for the complete Coswara pipeline and publication extras. The other notebooks are optional review/debug notebooks only.

Default behavior is conservative: expensive or optional stages are controlled by toggles, and existing outputs are skipped unless `FORCE_REBUILD = True`.

In [1]:
from pathlib import Path
import os
import subprocess
import sys
from datetime import datetime

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "covid_audio_btp").exists():
            return candidate
    raise FileNotFoundError("Could not find project root. Start Jupyter from the extracted covid_audio_btp folder or one of its subfolders.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")
print(f"Started: {datetime.now().isoformat(timespec='seconds')}")

Project root: C:\N Drive\Acads\BTP\covid_audio_btp_project_bundle_2026-05-26\covid_audio_btp
Python: C:\N Drive\Acads\BTP\covid_audio_btp_project_bundle_2026-05-26\covid_audio_btp\.venv\Scripts\python.exe
Started: 2026-06-12T04:04:50


## Configuration

Set only the paths and toggles you need. Keep `RUN_CNN = False` until the classical baseline pipeline is clean.

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
  PROJECT_ROOT = PROJECT_ROOT.parent

RAW_COSWARA_DIR = PROJECT_ROOT / "data/raw/coswara"
COUGHVID_RAW = None

FORCE_REBUILD = True

RUN_ENV_CHECK = True
RUN_LAYOUT_AUDIT = True

RUN_CORE_COSWARA = True
RUN_FEATURES = True
RUN_ML_BASELINES = True
RUN_CALIBRATION = True
RUN_FUSION = True
RUN_CNN = False
RUN_SHIFT_CHECKS = True
RUN_REPORT_ASSETS = True

RUN_PUBLICATION_EXTRAS = True
RUN_METADATA_BASELINE = True
RUN_QUALITY_WEIGHTED_FUSION = True
RUN_ABSTENTION = True
RUN_BOOTSTRAP_CI = True

RUN_COUGHVID_INDEX = False
RUN_COUGHVID_FEATURES = False
RUN_CROSS_DATASET = False
RUN_FEATURE_SHIFT_REPORT = False

RUN_PAPER_TABLES = True
RUN_PAIRED_MODEL_COMPARISON = True
RUN_CONFOUNDING_MATCHING = True
RUN_EXPERIMENT_MANIFEST = True

MIN_COUGH_DETECTED = 0.80
COUGHVID_FEATURE_MAX_ROWS = 25

CNN_EPOCHS = 50
CNN_BATCH_SIZE = 8

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Coswara:", RAW_COSWARA_DIR)
print("COUGHVID:", COUGHVID_RAW)
print("Coswara exists:", RAW_COSWARA_DIR.exists())

PROJECT_ROOT: C:\N Drive\Acads\BTP\covid_audio_btp_project_bundle_2026-05-26\covid_audio_btp
Coswara: C:\N Drive\Acads\BTP\covid_audio_btp_project_bundle_2026-05-26\covid_audio_btp\data\raw\coswara
COUGHVID: None
Coswara exists: True


## Runner

The helper below runs project scripts, checks required inputs, and skips completed outputs unless forced.

In [3]:
def p(path):
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path


def path_exists(path):
    return p(path).exists() and p(path).stat().st_size > 0


def missing(paths):
    return [str(path) for path in paths if not path_exists(path)]


def run_step(name, args, enabled=True, requires=None, creates=None, strict_requires=True, force=None):
    requires = requires or []
    creates = creates or []
    force = FORCE_REBUILD if force is None else force
    if not enabled:
        print(f"SKIP {name}: disabled")
        return False
    missing_inputs = missing(requires)
    if missing_inputs:
        message = f"SKIP {name}: missing required input(s): {missing_inputs}"
        if strict_requires:
            raise FileNotFoundError(message)
        print(message)
        return False
    if creates and not force and all(path_exists(path) for path in creates):
        print(f"SKIP {name}: outputs already exist")
        return False
    cmd = [str(item) for item in args]
    print()
    print(f"## {name}")
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()
    return True


CORE_ARTIFACTS = [
    "data/interim/coswara_index.csv",
    "data/processed/metadata_clean.csv",
    "data/interim/modality_availability.csv",
    "data/interim/split_manifest.csv",
    "data/processed/audio_quality.csv",
    "data/processed/features_mfcc.csv",
    "data/processed/spectrogram_index.csv",
    "data/outputs/metrics/ml_baseline_metrics.csv",
    "data/outputs/metrics/calibrated_branch_predictions.csv",
    "data/outputs/metrics/fusion_metrics.csv",
    "data/outputs/metrics/quality_weighted_fusion_metrics.csv",
    "reports/tables/paper_metric_table.csv",
    "data/outputs/metrics/paired_model_comparison.csv",
    "reports/tables/confounding_balance.csv",
    "reports/tables/feature_shift_report.csv",
    "reports/experiment_manifest.json",
]
print("Runner ready")

Runner ready


## Artifact Dashboard

This shows what already exists before the run.

In [4]:
try:
    import pandas as pd
    dashboard = pd.DataFrame(
        [{"path": path, "exists": path_exists(path), "size_bytes": p(path).stat().st_size if p(path).exists() else 0} for path in CORE_ARTIFACTS]
    )
    display(dashboard)
except Exception as exc:
    print(f"Dashboard unavailable before dependency install: {exc}")
    for path in CORE_ARTIFACTS:
        print(path, "OK" if path_exists(path) else "missing")

,path,exists,size_bytes
0,data/interim/coswara_index.csv,True,13280939
1,data/processed/metadata_clean.csv,True,13908498
2,data/interim/modality_availability.csv,True,179212
3,data/interim/split_manifest.csv,True,215868
4,data/processed/audio_quality.csv,False,0
5,data/processed/features_mfcc.csv,False,0
6,data/processed/spectrogram_index.csv,False,0
7,data/outputs/metrics/ml_baseline_metrics.csv,False,0
8,data/outputs/metrics/calibrated_branch_predict...,False,0
9,data/outputs/metrics/fusion_metrics.csv,False,0


## Stage 0-1: Environment, Index, Splits, Quality

This stage is mandatory before modeling. It prevents training on bad audio, bad labels, or participant leakage.

In [5]:
run_step("local preflight", [sys.executable, "scripts/00_local_preflight.py", "--coswara-dir", RAW_COSWARA_DIR], enabled=True, force=True)

run_step("environment check", [sys.executable, "scripts/00_check_environment.py"], enabled=RUN_ENV_CHECK, force=True)

if RUN_CORE_COSWARA and not RAW_COSWARA_DIR.exists():
    raise FileNotFoundError(f"Coswara not found. Put it at {RAW_COSWARA_DIR} or change RAW_COSWARA_DIR.")

run_step(
    "raw layout audit",
    [sys.executable, "scripts/00_inspect_dataset_layout.py", "--raw-dir", RAW_COSWARA_DIR, "--output", "reports/tables/coswara_layout_audit.csv"],
    enabled=RUN_CORE_COSWARA and RUN_LAYOUT_AUDIT,
    creates=["reports/tables/coswara_layout_audit.csv"],
)
run_step(
    "build Coswara index",
    [sys.executable, "scripts/01_build_coswara_index.py", "--raw-dir", RAW_COSWARA_DIR, "--output", "data/interim/coswara_index.csv"],
    enabled=RUN_CORE_COSWARA,
    creates=["data/interim/coswara_index.csv"],
)
run_step(
    "clean metadata",
    [sys.executable, "scripts/02_clean_metadata.py", "--index", "data/interim/coswara_index.csv", "--output", "data/processed/metadata_clean.csv", "--availability-output", "data/interim/modality_availability.csv", "--audit-output", "reports/tables/dataset_audit.csv"],
    enabled=RUN_CORE_COSWARA,
    requires=["data/interim/coswara_index.csv"],
    creates=["data/processed/metadata_clean.csv", "data/interim/modality_availability.csv"],
)
run_step(
    "participant splits",
    [sys.executable, "scripts/03_create_splits.py", "--metadata", "data/processed/metadata_clean.csv", "--output", "data/interim/split_manifest.csv", "--metadata-output", "data/processed/metadata_clean.csv", "--audit-output", "reports/tables/split_audit.csv"],
    enabled=RUN_CORE_COSWARA,
    requires=["data/processed/metadata_clean.csv"],
    creates=["data/interim/split_manifest.csv"],
)
run_step(
    "quality audit",
    [sys.executable, "scripts/04_quality_audit.py", "--metadata", "data/processed/metadata_clean.csv", "--output", "data/processed/audio_quality.csv", "--metadata-output", "data/processed/metadata_clean.csv", "--summary-output", "reports/tables/quality_summary.csv"],
    enabled=RUN_CORE_COSWARA,
    requires=["data/interim/split_manifest.csv", "data/processed/metadata_clean.csv"],
    creates=["data/processed/audio_quality.csv"],
)
run_step(
    "artifact validation",
    [sys.executable, "scripts/12_validate_artifacts.py", "--strict"],
    enabled=RUN_CORE_COSWARA,
    requires=["data/interim/coswara_index.csv", "data/processed/metadata_clean.csv", "data/processed/audio_quality.csv"],
    force=True,
)


## local preflight
$ C:\N Drive\Acads\BTP\covid_audio_btp_project_bundle_2026-05-26\covid_audio_btp\.venv\Scripts\python.exe scripts/00_local_preflight.py --coswara-dir C:\N Drive\Acads\BTP\covid_audio_btp_project_bundle_2026-05-26\covid_audio_btp\data\raw\coswara
Project root: C:\N Drive\Acads\BTP\covid_audio_btp_project_bundle_2026-05-26\covid_audio_btp
Python: C:\N Drive\Acads\BTP\covid_audio_btp_project_bundle_2026-05-26\covid_audio_btp\.venv\Scripts\python.exe
Coswara path: C:\N Drive\Acads\BTP\covid_audio_btp_project_bundle_2026-05-26\covid_audio_btp\data\raw\coswara
Notebook syntax: OK
Python syntax: OK
Required imports: OK
Coswara audio files discovered: 24716
Coswara CSV files discovered: 73

WARNINGS
- torch (torch): No module named 'torch'
- torchaudio (torchaudio): No module named 'torchaudio'

Preflight passed. It is safe to start the notebook pipeline.


## environment check
$ C:\N Drive\Acads\BTP\covid_audio_btp_project_bundle_2026-05-26\covid_audio_btp\.venv\Scripts\py

True

## Hard Gate

If this cell fails, inspect the optional review notebooks before training models.

In [6]:
import pandas as pd

from covid_audio_btp.notebook_utils import (
    assert_binary_labels_present,
    assert_no_participant_leakage,
    check_artifacts,
    read_optional_csv,
    stop_if_validation_errors,
)

metadata = pd.read_csv(p("data/processed/metadata_clean.csv"))
quality = pd.read_csv(p("data/processed/audio_quality.csv"))
issues = read_optional_csv("reports/tables/validation_issues.csv")

assert_binary_labels_present(metadata)
assert_no_participant_leakage(metadata)
if issues is not None:
    stop_if_validation_errors(issues)

quality_ok_rate = float((quality["quality_flag"] == "ok").mean()) if len(quality) else 0.0
print(f"quality ok rate: {quality_ok_rate:.3f}")
if quality_ok_rate < 0.50:
    raise AssertionError("Quality ok rate below 50%; review audio quality before modeling.")

reports/tables/validation_issues.csv: 1 rows x 3 columns
label gate passed: both positive and negative classes are present
leakage gate passed: 2746 participants appear in one split each
validation gate passed with warnings only
quality ok rate: 0.953


## Stage 2-5: Features, Baselines, Calibration, Fusion

This is the main Coswara modeling path. CNN is optional and disabled by default.

In [7]:
run_step(
    "feature extraction",
    [
        sys.executable,
        "scripts/05_extract_features.py",
        "--metadata",
        "data/processed/metadata_clean.csv",
        "--features-output",
        "data/processed/features_mfcc.csv",
        "--quality-mode",
        "all_samples",
        "--skip-spectrograms",
    ],
    enabled=RUN_FEATURES,
    requires=[
        "data/processed/metadata_clean.csv",
        "data/processed/audio_quality.csv",
    ],
    creates=["data/processed/features_mfcc.csv"],
    force=FORCE_REBUILD,
)
run_step(
    "classical ML baselines",
    [sys.executable, "scripts/06_train_ml_baselines.py", "--features", "data/processed/features_mfcc.csv"],
    enabled=RUN_ML_BASELINES,
    requires=["data/processed/features_mfcc.csv"],
    creates=["data/outputs/metrics/ml_baseline_metrics.csv", "data/outputs/metrics/ml_validation_metrics.csv", "data/outputs/metrics/ml_predictions_validation.csv", "data/outputs/metrics/ml_predictions_test.csv"],
)
run_step(
    "compact CNN cough branch",
    [sys.executable, "scripts/07_train_cnn.py", "--spectrogram-index", "data/processed/spectrogram_index.csv", "--modality", "cough", "--epochs", CNN_EPOCHS, "--batch-size", CNN_BATCH_SIZE],
    enabled=RUN_CNN,
    requires=["data/processed/spectrogram_index.csv"],
    creates=["data/outputs/metrics/cnn_metrics.csv", "data/outputs/metrics/cnn_logits_validation.csv", "data/outputs/metrics/cnn_logits_test.csv"],
)
run_step(
    "branch calibration",
    [sys.executable, "scripts/08_calibrate_branches.py", "--validation-predictions", "data/outputs/metrics/ml_predictions_validation.csv", "--test-predictions", "data/outputs/metrics/ml_predictions_test.csv", "--method", "platt"],
    enabled=RUN_CALIBRATION,
    requires=["data/outputs/metrics/ml_predictions_validation.csv", "data/outputs/metrics/ml_predictions_test.csv"],
    creates=["data/outputs/metrics/calibrated_branch_predictions.csv", "data/outputs/metrics/calibration_metrics.csv"],
)
run_step(
    "standard fusion",
    [sys.executable, "scripts/09_run_fusion.py", "--predictions", "data/outputs/metrics/calibrated_branch_predictions.csv", "--validation-metrics", "data/outputs/metrics/ml_validation_metrics.csv"],
    enabled=RUN_FUSION,
    requires=["data/outputs/metrics/calibrated_branch_predictions.csv", "data/outputs/metrics/ml_validation_metrics.csv"],
    creates=["data/outputs/metrics/fusion_predictions.csv", "data/outputs/metrics/fusion_metrics.csv"],
)
run_step(
    "subgroup and confounding checks",
    [sys.executable, "scripts/10_shift_and_confounding_checks.py", "--predictions", "data/outputs/metrics/fusion_predictions.csv", "--metadata", "data/processed/metadata_clean.csv"],
    enabled=RUN_SHIFT_CHECKS,
    requires=["data/outputs/metrics/fusion_predictions.csv", "data/processed/metadata_clean.csv"],
    creates=["data/outputs/metrics/subgroup_metrics.csv"],
)
run_step(
    "basic report assets",
    [sys.executable, "scripts/11_make_report_assets.py", "--metadata", "data/processed/metadata_clean.csv", "--predictions", "data/outputs/metrics/fusion_predictions.csv"],
    enabled=RUN_REPORT_ASSETS,
    requires=["data/processed/metadata_clean.csv"],
    creates=["reports/report_outline.md", "reports/slides_outline.md"],
)


## feature extraction
$ C:\N Drive\Acads\BTP\covid_audio_btp_project_bundle_2026-05-26\covid_audio_btp\.venv\Scripts\python.exe scripts/05_extract_features.py --metadata data/processed/metadata_clean.csv --features-output data/processed/features_mfcc.csv --quality-mode all_samples --skip-spectrograms
Wrote features: data\processed\features_mfcc.csv (19024 rows, 261 columns)


## classical ML baselines
$ C:\N Drive\Acads\BTP\covid_audio_btp_project_bundle_2026-05-26\covid_audio_btp\.venv\Scripts\python.exe scripts/06_train_ml_baselines.py --features data/processed/features_mfcc.csv
Trained dummy_most_frequent / cough: AUROC=0.5
Trained dummy_stratified / cough: AUROC=0.47945360126439385
Trained logistic_regression / cough: AUROC=0.785470760894107
Trained random_forest / cough: AUROC=0.8067396703544818
Trained dummy_most_frequent / breath: AUROC=0.5
Trained dummy_stratified / breath: AUROC=0.47945360126439385
Trained logistic_regression / breath: AUROC=0.75505757507338
Trained random_fo

True

## Stage 6: Publication Extras

These experiments support the publication-grade claims: metadata-only baseline, quality-weighted fusion, abstention, confidence intervals, COUGHVID external validation, and paper tables.

In [8]:
run_step(
    "metadata-only baseline",
    [sys.executable, "scripts/14_train_metadata_baseline.py", "--metadata", "data/processed/metadata_clean.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_METADATA_BASELINE,
    requires=["data/processed/metadata_clean.csv"],
    creates=["data/outputs/metrics/metadata_baseline_metrics.csv", "data/outputs/metrics/metadata_predictions_validation.csv", "data/outputs/metrics/metadata_predictions_test.csv"],
)
run_step(
    "quality-weighted fusion",
    [sys.executable, "scripts/16_run_quality_weighted_fusion.py", "--predictions", "data/outputs/metrics/calibrated_branch_predictions.csv", "--quality", "data/processed/audio_quality.csv", "--validation-metrics", "data/outputs/metrics/ml_validation_metrics.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_QUALITY_WEIGHTED_FUSION,
    requires=["data/outputs/metrics/calibrated_branch_predictions.csv", "data/processed/audio_quality.csv", "data/outputs/metrics/ml_validation_metrics.csv"],
    creates=["data/outputs/metrics/quality_weighted_fusion_predictions.csv", "data/outputs/metrics/quality_weighted_fusion_metrics.csv"],
)
run_step(
    "abstention analysis",
    [sys.executable, "scripts/17_abstention_analysis.py", "--predictions", "data/outputs/metrics/quality_weighted_fusion_predictions.csv", "--metadata", "data/processed/metadata_clean.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_ABSTENTION,
    requires=["data/outputs/metrics/quality_weighted_fusion_predictions.csv", "data/processed/metadata_clean.csv"],
    creates=["data/outputs/metrics/abstention_decisions.csv", "data/outputs/metrics/abstention_coverage_curve.csv"],
)
run_step(
    "bootstrap CI for quality-weighted fusion",
    [sys.executable, "scripts/15_bootstrap_prediction_metrics.py", "--predictions", "data/outputs/metrics/quality_weighted_fusion_predictions.csv", "--output", "data/outputs/metrics/quality_weighted_fusion_bootstrap_ci.csv", "--group-columns", "fusion_method"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_BOOTSTRAP_CI,
    requires=["data/outputs/metrics/quality_weighted_fusion_predictions.csv"],
    creates=["data/outputs/metrics/quality_weighted_fusion_bootstrap_ci.csv"],
)

if COUGHVID_RAW is not None:
    run_step(
        "COUGHVID index",
        [sys.executable, "scripts/13_build_coughvid_index.py", "--raw-dir", COUGHVID_RAW, "--output", "data/interim/coughvid_index.csv", "--min-cough-detected", MIN_COUGH_DETECTED],
        enabled=RUN_PUBLICATION_EXTRAS and RUN_COUGHVID_INDEX,
        creates=["data/interim/coughvid_index.csv"],
    )
else:
    print("SKIP COUGHVID index: COUGHVID_RAW is None")

feature_cmd = [sys.executable, "scripts/19_extract_coughvid_features.py", "--index", "data/interim/coughvid_index.csv", "--features-output", "data/processed/coughvid_features_mfcc.csv", "--quality-ok-only"]
if COUGHVID_FEATURE_MAX_ROWS is not None:
    feature_cmd += ["--max-rows", COUGHVID_FEATURE_MAX_ROWS]
run_step(
    "COUGHVID feature extraction",
    feature_cmd,
    enabled=RUN_PUBLICATION_EXTRAS and RUN_COUGHVID_FEATURES,
    requires=["data/interim/coughvid_index.csv"],
    creates=["data/processed/coughvid_features_mfcc.csv"],
    strict_requires=False,
)
run_step(
    "cross-dataset cough evaluation",
    [sys.executable, "scripts/18_cross_dataset_feature_eval.py", "--source-features", "data/processed/features_mfcc.csv", "--external-features", "data/processed/coughvid_features_mfcc.csv", "--modality", "cough", "--model-name", "logistic_regression"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_CROSS_DATASET,
    requires=["data/processed/features_mfcc.csv", "data/processed/coughvid_features_mfcc.csv"],
    creates=["data/outputs/metrics/cross_dataset_predictions.csv", "data/outputs/metrics/cross_dataset_metrics.csv"],
    strict_requires=False,
)
run_step(
    "paper metric tables",
    [sys.executable, "scripts/20_make_paper_tables.py"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_PAPER_TABLES,
    requires=["data/outputs/metrics/ml_baseline_metrics.csv"],
    creates=["reports/tables/paper_metric_table.csv", "reports/tables/paper_metric_table_raw.csv"],
    strict_requires=False,
)


## metadata-only baseline
$ C:\N Drive\Acads\BTP\covid_audio_btp_project_bundle_2026-05-26\covid_audio_btp\.venv\Scripts\python.exe scripts/14_train_metadata_baseline.py --metadata data/processed/metadata_clean.csv
Wrote metadata baseline metrics: data\outputs\metrics\metadata_baseline_metrics.csv
{'auroc': 0.9362665819277546, 'auprc': 0.9216161989052694, 'balanced_accuracy': 0.8905923082712426, 'f1': 0.8537248504622077, 'sensitivity': 0.8468176914778857, 'specificity': 0.9343669250645995, 'brier': 0.07405667612863413, 'ece': 0.07215307464262445, 'nll': 0.26826406759104116, 'threshold': 0.5, 'n_samples': 2862.0, 'model_name': 'metadata_logistic_regression', 'modality': 'metadata'}


## quality-weighted fusion
$ C:\N Drive\Acads\BTP\covid_audio_btp_project_bundle_2026-05-26\covid_audio_btp\.venv\Scripts\python.exe scripts/16_run_quality_weighted_fusion.py --predictions data/outputs/metrics/calibrated_branch_predictions.csv --quality data/processed/audio_quality.csv --validation-metrics

True

## Stage 7: Extra Publication Strengthening

These optional diagnostics make the paper stronger: paired model comparison, matched confounding subset, source-vs-external feature shift, and a reproducibility manifest.

In [9]:
run_step(
    "paired ML model comparison",
    [sys.executable, "scripts/21_paired_model_comparison.py", "--predictions", "data/outputs/metrics/calibrated_branch_predictions.csv", "--baseline-name", "logistic_regression", "--model-column", "model_name", "--group-columns", "modality", "--output", "data/outputs/metrics/paired_model_comparison.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_PAIRED_MODEL_COMPARISON,
    requires=["data/outputs/metrics/calibrated_branch_predictions.csv"],
    creates=["data/outputs/metrics/paired_model_comparison.csv"],
    strict_requires=False,
)
run_step(
    "confounding matched subset",
    [sys.executable, "scripts/22_confounding_matching.py", "--metadata", "data/processed/metadata_clean.csv", "--predictions", "data/outputs/metrics/quality_weighted_fusion_predictions.csv", "--covariates", "age_bucket", "gender"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_CONFOUNDING_MATCHING,
    requires=["data/processed/metadata_clean.csv"],
    creates=["data/processed/metadata_matched.csv", "reports/tables/confounding_balance.csv"],
    strict_requires=False,
)
run_step(
    "feature shift report",
    [sys.executable, "scripts/23_feature_shift_report.py", "--source-features", "data/processed/features_mfcc.csv", "--external-features", "data/processed/coughvid_features_mfcc.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_FEATURE_SHIFT_REPORT,
    requires=["data/processed/features_mfcc.csv", "data/processed/coughvid_features_mfcc.csv"],
    creates=["reports/tables/feature_shift_report.csv", "reports/tables/feature_shift_summary.csv"],
    strict_requires=False,
)
run_step(
    "experiment manifest",
    [sys.executable, "scripts/24_make_experiment_manifest.py", "--run-name", "covid_audio_publication_run"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_EXPERIMENT_MANIFEST,
    creates=["reports/experiment_manifest.json"],
)


## paired ML model comparison
$ C:\N Drive\Acads\BTP\covid_audio_btp_project_bundle_2026-05-26\covid_audio_btp\.venv\Scripts\python.exe scripts/21_paired_model_comparison.py --predictions data/outputs/metrics/calibrated_branch_predictions.csv --baseline-name logistic_regression --model-column model_name --group-columns modality --output data/outputs/metrics/paired_model_comparison.csv
Wrote paired comparison table: data\outputs\metrics\paired_model_comparison.csv (36 rows)


## confounding matched subset
$ C:\N Drive\Acads\BTP\covid_audio_btp_project_bundle_2026-05-26\covid_audio_btp\.venv\Scripts\python.exe scripts/22_confounding_matching.py --metadata data/processed/metadata_clean.csv --predictions data/outputs/metrics/quality_weighted_fusion_predictions.csv --covariates age_bucket gender
Wrote matched metadata: data\processed\metadata_matched.csv (11196 rows)
Wrote balance table: reports\tables\confounding_balance.csv
Wrote matched-subset metrics: data\outputs\metrics\matched_subse

True

## Final Dashboard

Use this at the end of the run to see exactly what was produced.

In [10]:
import pandas as pd

final_dashboard = pd.DataFrame(
    [{"path": path, "exists": path_exists(path), "size_bytes": p(path).stat().st_size if p(path).exists() else 0} for path in CORE_ARTIFACTS]
)
display(final_dashboard)

for path in [
    "data/outputs/metrics/ml_baseline_metrics.csv",
    "data/outputs/metrics/fusion_metrics.csv",
    "data/outputs/metrics/quality_weighted_fusion_metrics.csv",
    "reports/tables/paper_metric_table.csv",
    "data/outputs/metrics/paired_model_comparison.csv",
    "reports/tables/confounding_balance.csv",
    "reports/tables/feature_shift_report.csv",
    "reports/experiment_manifest.json",
]:
    if path_exists(path):
        print()
        print(path)
        display(pd.read_csv(p(path)).head(20))

print("Run finished")

,path,exists,size_bytes
0,data/interim/coswara_index.csv,True,15233187
1,data/processed/metadata_clean.csv,True,15958597
2,data/interim/modality_availability.csv,True,214297
3,data/interim/split_manifest.csv,True,245736
4,data/processed/audio_quality.csv,True,9408937
5,data/processed/features_mfcc.csv,True,90538316
6,data/processed/spectrogram_index.csv,False,0
7,data/outputs/metrics/ml_baseline_metrics.csv,True,2368
8,data/outputs/metrics/calibrated_branch_predict...,True,1492637
9,data/outputs/metrics/fusion_metrics.csv,True,541



data/outputs/metrics/ml_baseline_metrics.csv


,auroc,auprc,balanced_accuracy,f1,sensitivity,specificity,brier,ece,nll,threshold,n_samples,model_name,modality
0,0.500000,0.323899,0.500000,0.000000,0.000000,1.000000,0.323899,0.323899,4.474836,0.5,636.0,dummy_most_frequent,cough
1,0.479454,0.315672,0.479454,0.296117,0.296117,0.662791,0.455975,0.455975,6.299526,0.5,636.0,dummy_stratified,cough
2,0.785471,0.656269,0.724091,0.630522,0.762136,0.686047,0.195993,0.162325,0.583183,0.5,636.0,logistic_regression,cough
3,0.806740,0.718638,0.652664,0.484848,0.349515,0.955814,0.171125,0.118116,0.523498,0.5,636.0,random_forest,cough
4,0.500000,0.323899,0.500000,0.000000,0.000000,1.000000,0.323899,0.323899,4.474836,0.5,636.0,dummy_most_frequent,breath
5,0.479454,0.315672,0.479454,0.296117,0.296117,0.662791,0.455975,0.455975,6.299526,0.5,636.0,dummy_stratified,breath
6,0.755058,0.566053,0.695123,0.600760,0.766990,0.623256,0.204893,0.142385,0.591834,0.5,636.0,logistic_regression,breath
7,0.798600,0.671739,0.673956,0.535385,0.422330,0.925581,0.169116,0.070503,0.517116,0.5,636.0,random_forest,breath
8,0.500000,0.323899,0.500000,0.000000,0.000000,1.000000,0.323899,0.323899,4.474836,0.5,1590.0,dummy_most_frequent,speech
9,0.507415,0.327248,0.507415,0.333009,0.332039,0.682791,0.430818,0.430818,5.951966,0.5,1590.0,dummy_stratified,speech



data/outputs/metrics/fusion_metrics.csv


,auroc,auprc,balanced_accuracy,f1,sensitivity,specificity,brier,ece,nll,threshold,n_samples,fusion_method
0,0.878573,0.838432,0.793407,0.713043,0.796117,0.790698,0.189421,0.14544,0.564121,0.335698,318.0,uniform_mean
1,0.878167,0.841995,0.810793,0.745098,0.737864,0.883721,0.189074,0.14881,0.563384,0.349269,318.0,validation_weighted_auprc


...


reports/tables/confounding_balance.csv


,covariate,type,n_positive,n_negative,max_abs_standardized_difference,details_json
0,age_bucket,categorical,5598,5598,0.0,"{""30-44"":0.0,""45-59"":0.0,""60+"":0.0,""<30"":0.0}"
1,gender,categorical,5598,5598,0.0,"{""female"":0.0,""male"":0.0,""other"":0.0}"



reports/experiment_manifest.json


,{
"""created_at_utc"": ""2026-06-11T23:10:42.191094+00:00""",NaN
"""platform"": ""Windows-11-10.0.26200-SP0""",NaN
"""python_executable"": ""C:\\N Drive\\Acads\\BTP\\covid_audio_btp_project_bundle_2026-05-26\\covid_audio_btp\\.venv\\Scripts\\python.exe""",NaN
"""config"": {",NaN
"""run_name"": ""covid_audio_publication_run""",NaN
"""seed"": 42",NaN
},NaN
"""packages"": {",NaN
"""python"": ""3.12.10""",NaN
"""numpy"": ""2.4.6""",NaN


Run finished
